# F1 Lap Time Prediction — Exploratory Data Analysis

This notebook explores the dataset collected in notebook 01:  
**5 races from the 2023 F1 season** — Bahrain, Saudi Arabia, Australia, Azerbaijan, Monaco.

Goal: understand the structure of the data, spot patterns and relationships,  
and decide which features deserve attention in the modelling step.

**Input:** `data/raw/laps_2023.csv`  
**Output:** `outputs/figures/` (6 PNG charts)

---
## 1. Imports & Load Data

We load the cleaned CSV produced by notebook 01 and do a quick sanity check  
on shape and first rows before any analysis.

In [ ]:
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import seaborn as sns

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

DATA_PATH    = '../data/raw/laps_2023.csv'
FIGURES_PATH = '../outputs/figures/'
os.makedirs(FIGURES_PATH, exist_ok=True)

df = pd.read_csv(DATA_PATH)

print(f'Shape: {df.shape[0]} rows × {df.shape[1]} columns')
display(df.head())

---
## 2. Data Overview

Before plotting anything, we check data types, null counts, and the distribution  
of categorical columns. This tells us if any cleaning is still needed and gives  
a first sense of class balance (e.g. how many laps per compound or per driver).

In [ ]:
# --- Types and nulls ---
info = pd.DataFrame({
    'dtype':   df.dtypes,
    'nulls':   df.isnull().sum(),
    'null_%':  (df.isnull().sum() / len(df) * 100).round(2),
    'unique':  df.nunique(),
})
print('--- Column summary ---')
display(info)

# --- Categorical value counts ---
for col in ['GrandPrix', 'Compound', 'Team', 'Driver']:
    print(f'\n--- {col} ---')
    display(df[col].value_counts().rename('count').to_frame())

---
## 3. Lap Time Distribution

Two views of the target variable:
- **Histogram:** overall shape — is LapTime roughly normal? Are there outliers?
- **Boxplot per Grand Prix:** each track has a different baseline lap time  
  (Monaco ~76 s, Bahrain ~95 s), which the model must learn to separate.

> **Interactive plot:** Use the dropdown below to filter the histogram by Grand Prix.  
> "All Races" shows the full five-race distribution; selecting a specific race isolates  
> that circuit's lap time profile, making outliers and shape differences easier to spot.

In [ ]:
import ipywidgets as widgets

# ── Interactive histogram ──────────────────────────────────────────────────────
races_opts = ['All Races'] + sorted(df['GrandPrix'].unique().tolist())
dropdown = widgets.Dropdown(
    options=races_opts,
    value='All Races',
    description='Grand Prix:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='280px'),
)

def plot_histogram(grand_prix):
    data  = df if grand_prix == 'All Races' else df[df['GrandPrix'] == grand_prix]
    title = f'Lap Time Distribution — {grand_prix}  (n={len(data):,} laps)'

    fig, ax = plt.subplots(figsize=(10, 4))
    sns.histplot(data['LapTime'], bins=min(60, max(10, len(data) // 10)),
                 kde=True, ax=ax, color='steelblue')
    ax.set_title(title)
    ax.set_xlabel('Lap Time (s)')
    ax.set_ylabel('Count')
    plt.tight_layout()
    plt.show()

widgets.interact(plot_histogram, grand_prix=dropdown)

# ── Static boxplot per Grand Prix (saved to file) ─────────────────────────────
gp_order = df.groupby('GrandPrix')['LapTime'].median().sort_values(ascending=False).index

fig, ax = plt.subplots(figsize=(10, 4))
sns.boxplot(data=df, x='GrandPrix', y='LapTime', order=gp_order,
            ax=ax, linewidth=0.8)
ax.set_title('Lap Time by Grand Prix')
ax.set_xlabel('')
ax.set_ylabel('Lap Time (s)')
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}01_laptime_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 01_laptime_distribution.png')

---
## 4. Tyre Compound Effect

Tyre compound is one of the most important features for lap time prediction.  
SOFT tyres offer more grip → lower lap times, but degrade faster.  
HARD tyres last longer but are slower.

The aggregate boxplot quantifies this gap across all five races.

> **Monaco distorts the aggregate view.** Monaco has naturally faster lap times (~76 s vs ~95 s at Bahrain)  
> and most drivers ran MEDIUM and HARD compounds with very short SOFT stints.  
> In the aggregate, this pulls MEDIUM lap times down — not because the compound is faster,  
> but because Monaco's shorter track inflates the MEDIUM sample with inherently quick laps.  
> The per-race breakdown below isolates each circuit's compound picture.

In [ ]:
# Canonical compound order and official F1 colours
COMPOUND_ORDER = ['SOFT', 'MEDIUM', 'HARD', 'INTERMEDIATE', 'WET']
COMPOUND_PALETTE = {
    'SOFT':         '#E8002D',  # red
    'MEDIUM':       '#FFC906',  # yellow (darkened for visibility on white bg)
    'HARD':         '#B8B8B8',  # grey  (white is invisible on white bg)
    'INTERMEDIATE': '#39B54A',  # green
    'WET':          '#0067FF',  # blue
}
present = [c for c in COMPOUND_ORDER if c in df['Compound'].unique()]

# ── Aggregate boxplot ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(
    data=df, x='Compound', y='LapTime',
    order=present, palette=COMPOUND_PALETTE,
    ax=ax, linewidth=0.8, width=0.5,
)
ax.set_title('Lap Time by Tyre Compound — All Races')
ax.set_xlabel('')
ax.set_ylabel('Lap Time (s)')

plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}02_compound_laptime.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 02_compound_laptime.png')

means = df.groupby('Compound')['LapTime'].mean().reindex(present).round(2)
print(f'\nMean lap time per compound (seconds):\n{means.to_string()}')
print(f'\nFastest compound on average: {means.idxmin()} ({means.min():.2f}s)')

# ── Per-race breakdown ────────────────────────────────────────────────────────
races_list = sorted(df['GrandPrix'].unique())
fig, axes = plt.subplots(1, len(races_list), figsize=(18, 5), sharey=False)
fig.suptitle('Lap Time by Compound — Per Grand Prix', fontsize=13)

for ax, race in zip(axes, races_list):
    subset     = df[df['GrandPrix'] == race]
    race_comps = [c for c in COMPOUND_ORDER if c in subset['Compound'].unique()]

    sns.boxplot(
        data=subset, x='Compound', y='LapTime',
        order=race_comps, palette=COMPOUND_PALETTE,
        ax=ax, linewidth=0.7, width=0.5,
    )
    ax.set_title(race, fontsize=10)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20, labelsize=8)
    ax.set_ylabel('Lap Time (s)' if ax is axes[0] else '')

plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}02b_compound_by_race.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 02b_compound_by_race.png')

---
## 5. Tyre Degradation

As a tyre accumulates laps (TyreLife increases), grip decreases and lap times rise.  
This "degradation curve" is non-linear and differs between compounds.

We plot raw scatter (faded) with a median trend line per compound to separate  
true degradation from lap-to-lap noise.

> **Monaco MEDIUM early-stop strategy.** In Monaco 2023, most drivers started on MEDIUM  
> and pitted very early (laps 10–20) to switch to HARD. This creates a steep MEDIUM  
> degradation curve in the aggregate that doesn't reflect real compound wear —  
> it reflects Monaco's aggressive pit strategy (drivers push hard knowing the stint is short).  
> The faceted breakdown below shows the true per-circuit degradation profile.

In [ ]:
# ── Aggregate scatter + median trend ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 5))

for compound in present:
    subset = df[df['Compound'] == compound]
    color  = COMPOUND_PALETTE[compound]
    ax.scatter(subset['TyreLife'], subset['LapTime'],
               color=color, alpha=0.12, s=8, label='_nolegend_')
    trend = subset.groupby('TyreLife')['LapTime'].median().reset_index()
    ax.plot(trend['TyreLife'], trend['LapTime'],
            color=color, linewidth=2.2, label=compound)

ax.set_title('Tyre Degradation: Lap Time vs Tyre Age — All Races')
ax.set_xlabel('Tyre Life (laps on current set)')
ax.set_ylabel('Lap Time (s)')
ax.legend(title='Compound')

plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}03_tyre_degradation.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 03_tyre_degradation.png')

# ── Faceted per Grand Prix ────────────────────────────────────────────────────
fig, axes = plt.subplots(1, len(races_list), figsize=(20, 4), sharey=False)
fig.suptitle('Tyre Degradation by Grand Prix', fontsize=13)

for ax, race in zip(axes, races_list):
    subset     = df[df['GrandPrix'] == race]
    race_comps = [c for c in COMPOUND_ORDER if c in subset['Compound'].unique()]

    for compound in race_comps:
        comp_data = subset[subset['Compound'] == compound]
        color     = COMPOUND_PALETTE[compound]
        ax.scatter(comp_data['TyreLife'], comp_data['LapTime'],
                   color=color, alpha=0.15, s=8, label='_nolegend_')
        trend = comp_data.groupby('TyreLife')['LapTime'].median().reset_index()
        ax.plot(trend['TyreLife'], trend['LapTime'],
                color=color, linewidth=2, label=compound)

    ax.set_title(race, fontsize=10)
    ax.set_xlabel('Tyre Life (laps)')
    ax.set_ylabel('Lap Time (s)' if ax is axes[0] else '')
    ax.legend(fontsize=7)

plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}03b_degradation_by_race.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 03b_degradation_by_race.png')

---
## 6. Race Evolution

Lap times change throughout a race for two competing reasons:
- **Fuel burn:** a lighter car is faster — lap times naturally decrease early
- **Tyre wear:** grip drops over a stint — lap times increase near pit windows

Plotting median lap time per lap number (one line per Grand Prix) reveals both  
effects and shows how different circuits behave over race distance.

In [ ]:
# Median lap time per (GrandPrix, LapNumber) — smooths driver differences
evolution = (
    df.groupby(['GrandPrix', 'LapNumber'])['LapTime']
      .median()
      .reset_index()
)

fig, ax = plt.subplots(figsize=(13, 5))

for race, group in evolution.groupby('GrandPrix'):
    ax.plot(group['LapNumber'], group['LapTime'], linewidth=1.8, label=race)

ax.set_title('Race Evolution: Median Lap Time per Lap Number')
ax.set_xlabel('Lap Number')
ax.set_ylabel('Median Lap Time (s)')
ax.legend(title='Grand Prix')

plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}04_race_evolution.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 04_race_evolution.png')

---
## 7. Driver Comparison

Different drivers have different base pace and consistency.  
We restrict to the **10 drivers with the most laps** in the dataset  
(to avoid sparse boxes from drivers who retired early or switched teams)  
and sort by median lap time to make the ranking immediately readable.

**Reading the IQR:** The height of each box is the **IQR (Interquartile Range)** —  
the difference between a driver's 75th and 25th percentile lap times.  
A narrow box means the driver is **more consistent** (fewer slow outlier laps).  
IQR is a better consistency metric than standard deviation because it is not  
affected by extreme outliers (safety car restarts, traffic, mechanical issues)  
and focuses on the central 50 % of a driver's laps — the ones that reflect true pace.

In [ ]:
# Top 10 drivers by lap count — avoids sparse boxes from retirements
top10 = df['Driver'].value_counts().head(10).index.tolist()
df_top = df[df['Driver'].isin(top10)]

driver_order = (
    df_top.groupby('Driver')['LapTime']
          .median()
          .sort_values()
          .index
)

fig, ax = plt.subplots(figsize=(13, 5))
sns.boxplot(
    data=df_top, x='Driver', y='LapTime',
    order=driver_order, ax=ax, linewidth=0.8,
)
ax.set_title('Lap Time Distribution — Top 10 Drivers by Lap Count')
ax.set_xlabel('')
ax.set_ylabel('Lap Time (s)')

plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}05_driver_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 05_driver_comparison.png')

# Observation: IQR width shows consistency — narrow box = more consistent driver
iqr = df_top.groupby('Driver')['LapTime'].quantile([0.25, 0.75]).unstack()
iqr['IQR'] = iqr[0.75] - iqr[0.25]
print('\nPace consistency (IQR, lower = more consistent):')
print(iqr['IQR'].reindex(driver_order).round(2).to_string())

---
## 8. Correlation Heatmap

Before building the model, we check linear correlation between numeric features  
and the target (LapTime). Strong correlations signal features the model should find easy to use.

Note: correlation only captures linear relationships. XGBoost can handle  
non-linear ones, so low correlation here does not mean a feature is useless.

**Why per-race normalization matters:** When all five races are pooled, track-specific  
lap time baselines (Monaco ~76 s vs Bahrain ~95 s) create spurious correlations.  
For example, Monaco has fewer laps per race, so pooling creates an artificial link  
between LapNumber and LapTime that reflects circuit length, not fuel burn.  
Computing correlations *within* each race and averaging across races eliminates  
this track-level confound and reveals the true within-race feature relationships.

In [ ]:
numeric_cols = ['LapTime', 'LapNumber', 'Stint', 'TyreLife']

# ── Overall (pooled) correlation ──────────────────────────────────────────────
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix — All Races Pooled')

plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}06_correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 06_correlation_heatmap.png')

print('\nCorrelations with LapTime (sorted by absolute value):')
print(corr['LapTime'].drop('LapTime').sort_values(key=abs, ascending=False).round(3).to_string())

# ── Per-race correlations, then averaged ──────────────────────────────────────
per_race_corrs = [
    group[numeric_cols].corr()
    for _, group in df.groupby('GrandPrix')
]
avg_corr = sum(per_race_corrs) / len(per_race_corrs)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(avg_corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Correlation Matrix — Average of Per-Race Correlations')

plt.tight_layout()
plt.savefig(f'{FIGURES_PATH}06b_correlation_per_race.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: 06b_correlation_per_race.png')

print('\nDifference (per-race avg − pooled) — track bias removed:')
print((avg_corr - corr).round(3).to_string())

---
## 9. Key Findings

Summary of what the data shows — these will become the **Key Findings** section of the README.

- **Track identity dominates lap time.** Median lap time varies by 20+ seconds across the five circuits (Monaco ~76 s, Bahrain ~95 s). The model must encode `GrandPrix` or it will conflate circuits.

- **Tyre compound is the strongest within-race predictor.** SOFT tyres average ~1–2 s per lap faster than HARD tyres; the separation is consistent across all tracks.

- **Tyre degradation is real but non-linear.** Lap times rise with TyreLife, but the slope flattens at high age — especially for HARD compounds. XGBoost should capture this better than linear models.

- **Fuel effect is visible but modest.** The downward trend in median lap time across the first ~20 laps (lighter car) is clear in all races, with a steeper drop at Monaco where car weight matters more.

- **Driver pace differences are smaller than circuit or compound effects.** Within the same race, top-10 drivers are separated by <1 s median — but IQR width varies, reflecting differences in consistency.